In [3]:
import pandas as pd
import numpy as np
import os
import math
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

# Ensure output directory exists
os.makedirs('outputs', exist_ok=True)

# ==========================================
# 0. DATA LOADING & SETUP
# ==========================================
# IMPORTANT: Update this path to point to your actual data file
try:
    df = pd.read_csv('data/records.csv')
except FileNotFoundError:
    print("ERROR: data/records.csv not found. Please ensure the file exists.")
    exit()

# Assuming target column is named 'y' and is 1/0. Adjust if needed.
target_col = 'loan_default' 
train_df = df.iloc[:560].copy()
test_df = df.iloc[560:800].copy() # 240 rows

y_train = train_df[target_col]
y_test = test_df[target_col]

# ==========================================
# 4.1 DECISION TREE SPLITS
# ==========================================
def calc_entropy(p):
    if p == 0 or p == 1: return 0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def calc_gini(p):
    return 1 - p*2 - (1 - p)*2

p_root = y_train.mean()
root_ent = calc_entropy(p_root)
root_gini = calc_gini(p_root)

# Define split conditions (S1-S4) based on the prompt text
# (You may need to tweak these exact conditions based on the prompt's precise wording)
splits = {
    'S1': lambda x: x['irregular_salary_flag'] == 1,
    'S2': lambda x: x['autopay_flag'] == 1,
    'S3': lambda x: x['credit_utilisation_ratio'] > 0.5,
    'S4': lambda x: x['account_age_months'] <= 24
}

split_results = []
for name, condition in splits.items():
    left_mask = condition(train_df)
    right_mask = ~left_mask
    
    left_y = train_df[left_mask][target_col]
    right_y = train_df[right_mask][target_col]
    
    p_left = left_y.mean() if len(left_y) > 0 else 0
    p_right = right_y.mean() if len(right_y) > 0 else 0
    
    w_left = len(left_y) / len(train_df)
    w_right = len(right_y) / len(train_df)
    
    # Entropy Gain
    w_ent = (w_left * calc_entropy(p_left)) + (w_right * calc_entropy(p_right))
    info_gain = root_ent - w_ent
    
    # Gini Gain
    w_gini = (w_left * calc_gini(p_left)) + (w_right * calc_gini(p_right))
    gini_gain = root_gini - w_gini
    
    split_results.append({
        'split': name, 'n': len(train_df), 'positives': y_train.sum(),
        'positives_no': (~y_train.astype(bool)).sum(), 
        'info_gain': info_gain, 'gini_gain': gini_gain
    })

splits_df = pd.DataFrame(split_results)
best_split_info = splits_df.loc[splits_df['info_gain'].idxmax()]
best_split_gini = splits_df.loc[splits_df['gini_gain'].idxmax()]['split']

# Format outputs as requested (checkpoint 0.7462 for root_entropy)
out_splits = pd.DataFrame([{
    'root_entropy': root_ent, 'root_gini': root_gini,
    'S1': splits_df[splits_df['split']=='S1']['info_gain'].values[0],
    'S2': splits_df[splits_df['split']=='S2']['info_gain'].values[0],
    'S3': splits_df[splits_df['split']=='S3']['info_gain'].values[0],
    'S4': splits_df[splits_df['split']=='S4']['info_gain'].values[0],
    'best_split_by_info_gain': best_split_info['split'],
    'best_split_by_gini_gain': best_split_gini
}])
out_splits.to_csv('outputs/part4_splits.csv', index=False)

# ==========================================
# 4.2 MODEL VOTING
# ==========================================
# 1. Train Logistic Regression (Assuming X_1 to X_7 are features)
features = [c for c in train_df.columns if c != target_col]
X_train, X_test = train_df[features], test_df[features]
cols_to_drop = [c for c in X_train.columns if 'id' in c.lower() or 'unnamed' in c.lower()]
X_train = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

# 2. Convert remaining categorical string columns to dummy variables (One-Hot Encoding)
# This turns text columns into 0s and 1s so the model can understand them
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# 3. Align Train and Test sets
# This ensures both have the exact same columns (in case one set has a category the other doesn't)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# 4. Ensure all data is numeric (float)
# This forces any remaining weird values to numeric, turning errors into NaNs
X_train = X_train.apply(pd.to_numeric, errors='coerce').fillna(0)
X_test = X_test.apply(pd.to_numeric, errors='coerce').fillna(0)

# --- END OF FIX ---

# Now run your models:
#log_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
#prob_log = log_model.predict_proba(X_test)[:, 1]
log_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
prob_log = log_model.predict_proba(X_test)[:, 1]

# 2. Train KNN (You need to define K from Task 2.7, assuming K=5 here)
K = 5 
knn_model = KNeighborsClassifier(n_neighbors=K).fit(X_train, y_train)
prob_knn = knn_model.predict_proba(X_test)[:, 1]

# 3. Stump Probability (Using the best split from 4.1)
best_split_condition = splits[best_split_info['split']]
# Calculate probability from training data for the stump
train_left_mask = best_split_condition(train_df)
stump_prob_left = y_train[train_left_mask].mean()
stump_prob_right = y_train[~train_left_mask].mean()

# Apply to test set
test_left_mask = best_split_condition(test_df)
prob_stump = np.where(test_left_mask, stump_prob_left, stump_prob_right)

# 4. Voting Logic
# Checkpoint from prompt: positive when probability >= 0.3
pred_log = (prob_log >= 0.3).astype(int)
pred_knn = (prob_knn >= 0.3).astype(int)
pred_stump = (prob_stump >= 0.3).astype(int)

# Hard Vote (at least 2 out of 3)
pred_hard = ((pred_log + pred_knn + pred_stump) >= 2).astype(int)

# Soft Vote (average probability >= 0.3)
prob_soft = (prob_log + prob_knn + prob_stump) / 3
pred_soft = (prob_soft >= 0.3).astype(int)

# Helper function for metrics
def get_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return tp, tn, fp, fn, precision, recall, f1

voting_results = []
models = {
    'logistic': pred_log,
    'knn': pred_knn,
    'stump': pred_stump,
    'hard_vote': pred_hard,
    'soft_vote': pred_soft
}

for name, preds in models.items():
    tp, tn, fp, fn, prec, rec, f1 = get_metrics(y_test, preds)
    voting_results.append({
        'model': name, 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'precision': round(prec, 4), 'recall': round(rec, 4), 'f1': round(f1, 4)
    })

pd.DataFrame(voting_results).to_csv('outputs/part4_voting.csv', index=False)

# ==========================================
# 4.3 ADABOOST BY HAND
# ==========================================
# Data from prompt image
y_ada = np.array([1, 1, 1, -1, -1, -1, -1, 1])
h1_ada = np.array([1, 1, -1, 1, -1, -1, 1, -1])
h2_ada = np.array([1, -1, 1, -1, -1, 1, 1, -1])

def run_adaboost_round(y, h, current_weights):
    # 1. Error
    misclassified = (h != y)
    error = np.sum(current_weights[misclassified])
    
    # Avoid division by zero
    if error == 0: error = 1e-10 
    if error == 1: error = 1 - 1e-10
    
    # 2. Alpha
    alpha = 0.5 * np.log((1 - error) / error)
    
    # 3. Update Weights
    unnormalized_w = current_weights * np.exp(-alpha * y * h)
    new_weights = unnormalized_w / np.sum(unnormalized_w)
    
    return error, alpha, new_weights, unnormalized_w

# Round 1
w1 = np.ones(8) / 8
err1, alpha1, norm_w1, unnorm_w1 = run_adaboost_round(y_ada, h1_ada, w1)

# Round 2
err2, alpha2, norm_w2, unnorm_w2 = run_adaboost_round(y_ada, h2_ada, norm_w1)

# Ensemble Accuracy
ensemble_pred = np.sign(alpha1 * h1_ada + alpha2 * h2_ada)
acc = np.mean(ensemble_pred == y_ada)

ada_rows = []
for i in range(8):
    ada_rows.append({
        'round': 1, 'point': i+1, 'weight_before': round(w1[i], 4),
        'weight_unnormalised': round(unnorm_w1[i], 4), 
        'weight_after': round(norm_w1[i], 4),
        'round1_error': round(err1, 4), 'round1_alpha': round(alpha1, 4)
    })
    ada_rows.append({
        'round': 2, 'point': i+1, 'weight_before': round(norm_w1[i], 4),
        'weight_unnormalised': round(unnorm_w2[i], 4), 
        'weight_after': round(norm_w2[i], 4),
        'round2_error': round(err2, 4), 'round2_alpha': round(alpha2, 4)
    })

pd.DataFrame(ada_rows).to_csv('outputs/part4_adaboost.csv', index=False)
print(f"AdaBoost Ensemble Accuracy: {acc}")
# Note: The prompt asks to write ensemble_accuracy into the CSV. 
# You can append this as a column or separate row depending on strict grading rules.

# ==========================================
# 4.4 OUT-OF-BAG FRACTION
# ==========================================
oob_fraction = (1 - (1/560))**560
print(f"OOB Fraction: {oob_fraction}")
# You will need to manually add this to your README or a separate text file as requested.

# ==========================================
# 4.5 NEURAL NETWORK ARCHITECTURE
# ==========================================
# Model A calculations
# Conv1: (28x28x1) -> F=5, C=6, P=2, S=1 -> Output: 28x28x6. Params: (5*5*1 + 1)*6 = 156
# Pool1: 28x28x6 -> 2x2, S=2 -> Output: 14x14x6. Params: 0
# Conv2: 14x14x6 -> F=5, C=16, P=2, S=1 -> Output: 14x14x16. Params: (5*5*6 + 1)*16 = 2416
# Pool2: 14x14x16 -> 2x2, S=2 -> Output: 7x7x16. Params: 0
# Flatten: 7*7*16 = 784
# Dense1: 784 -> 32. Params: (784*32) + 32 = 25120
# Output: 32 -> 10. Params: (32*10) + 10 = 330

# Model B calculations
# W_xh: 128 x 64 (192 params? No, 128*64 + 128 = 8320)
# W_hh: 128 x 128 (128*128 + 128 = 16512)
# W_hy: 16 x 128 (16*128 + 16 = 2064)

arch_rows = [
    {'model': 'A', 'layer': 'conv1', 'output_shape': '28x28x6', 'params': 156},
    {'model': 'A', 'layer': 'pool1', 'output_shape': '14x14x6', 'params': 0},
    {'model': 'A', 'layer': 'conv2', 'output_shape': '14x14x16', 'params': 2416},
    {'model': 'A', 'layer': 'pool2', 'output_shape': '7x7x16', 'params': 0},
    {'model': 'A', 'layer': 'fc_1', 'output_shape': '32', 'params': 25120},
    {'model': 'A', 'layer': 'output', 'output_shape': '10', 'params': 330},
    {'model': 'B', 'layer': 'W_xh', 'output_shape': '128', 'params': 8320},
    {'model': 'B', 'layer': 'W_hh', 'output_shape': '128', 'params': 16512},
    {'model': 'B', 'layer': 'W_hy', 'output_shape': '16', 'params': 2064}
]
pd.DataFrame(arch_rows).to_csv('outputs/part4_architecture.csv', index=False)

print("All CSV files generated in the 'outputs' directory.")

AdaBoost Ensemble Accuracy: 0.0
OOB Fraction: 0.36755073277355804
All CSV files generated in the 'outputs' directory.
